# Group 9 Project: UC New Bearcat Helper


*Install packages*

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

*Imports and Set-up*

In [ ]:
from dotenv import load_dotenv
import os
import re

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr

# Load credentials
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

# Configure OpenAI SDK
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

*1. UC Directory Agent and Tools*

In [ ]:
reader = PdfReader("notes/directory.pdf")
raw_pages = []
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_pages.append(text)

# The PDF inserts a newline between almost every word, so collapse whitespace first.
directory_text = " ".join(" ".join(raw_pages).split())

_department_pattern = re.compile(
    r"(?P<num>\d+)\.\s+"
    r"(?P<name>.+?)\s+"
    r"●\s+Topics:\s+(?P<topics>.+?)\s+"
    r"●\s+Phone number:\s+(?P<phone>.+?)\s+"
    r"●\s+Email:\s+(?P<email>.+?)\s+"
    r"●\s+Website:\s+(?P<website>\S+)",
    re.IGNORECASE,
)

directory_entries = []
for match in _department_pattern.finditer(directory_text):
    directory_entries.append(
        {
            "name": match.group("name").strip(),
            "topics": match.group("topics").strip(),
            "phone": match.group("phone").strip(),
            "email": match.group("email").strip(),
            "website": match.group("website").strip().rstrip("."),
        }
    )

print(f"Loaded {len(directory_entries)} departments from the UC directory.")
for entry in directory_entries:
    print("-", entry["name"])

In [ ]:
def _format_directory_entry(entry: dict) -> str:
    return (
        f"Department: {entry['name']}\n"
        f"Topics: {entry['topics']}\n"
        f"Phone: {entry['phone']}\n"
        f"Email: {entry['email']}\n"
        f"Website: {entry['website']}"
    )


def _entry_matches(entry: dict, query: str) -> bool:
    haystack = f"{entry['name']} {entry['topics']}".lower()
    if query in haystack:
        return True
    words = [word for word in re.split(r"[^a-z0-9]+", query) if len(word) > 2]
    if not words:
        return False
    return all(word in haystack for word in words)


@function_tool
def uc_directory_search(topic: str) -> str:
    """Look up a UC department in the campus directory and return
    its phone number, email, and website.

    Args:
        topic: A department name or keywords from the user's question
            (for example "housing", "admissions", or "parking pass").
    """
    print(f"Tool called to look up the department for: {topic}")
    query = topic.strip().lower()
    if not query:
        return "Please provide a department name or topic to search."

    matches = [entry for entry in directory_entries if _entry_matches(entry, query)]
    if not matches:
        words = [word for word in re.split(r"[^a-z0-9]+", query) if len(word) > 3]
        for entry in directory_entries:
            haystack = f"{entry['name']} {entry['topics']}".lower()
            if any(word in haystack for word in words):
                matches.append(entry)

    if not matches:
        return (
            f"We could not find anything related to '{topic}' in the directory. "
            "Direct the student to the Automated Directory Attendant at (513) 556-6000."
        )

    return "\n\n".join(_format_directory_entry(entry) for entry in matches)

In [ ]:
uc_directory_instructions = f"""{RECOMMENDED_PROMPT_PREFIX}
You are a helpful assistant that can search the UC directory for information.

Always call the uc_directory_search tool before answering questions about
department contacts, phone numbers, emails, or websites. Pass department names
or topic keywords from the user's question (for example "housing" or "financial aid").
Use only the phone, email, and website returned by the tool. Do not invent contact details.

If the tool finds a department, return its contact information clearly.
If the tool does not find a match, direct the student to the Automated Directory
Attendant at (513) 556-6000.
"""
uc_directory_agent = Agent(
    name="UC Directory Agent",
    instructions=uc_directory_instructions,
    model=azure_model,
    tools=[uc_directory_search],
)

*2. Admissions Agent and Tools*

In [ ]:
#Application deadlines

APP_DEADLINES = {
    "FY_APP_DEADLINES": {
        "Early Action": {
            "fall semester": "November 1"
        },
        "Rolling Admission": {
            "fall semester": "March 1",
            "spring semester": "November 1",
            "summer semester": "March 1"
        },
        "Confirmation": {
            "fall semester": "May 1",
            "spring semester": "December 1",
            "summer semester": "April 1"
        }
    },

    "TRANSFER_APP_DEADLINES": {
        "Rolling Admission": {
            "fall semester": "July 1",
            "spring semester": "November 1",
            "summer semester": "March 1"
        },
        "Confirmation": {
            "fall semester": "August 1",
            "spring semester": "December 1",
            "summer semester": "April 1"
        }
    }
}


In [ ]:
#Look up application deadlines
@function_tool
def lookup_application_tool(
    application_type: str,
    admission_type: str,
    semester: str
) -> str:
    """Look up an application deadline.

    Args:
        application_type: First-year or transfer application.
        admission_type: Early Action, Rolling Admission, or Confirmation.
        semester: Fall, spring, or summer.
    """

    print(f"Looking up application deadline: {application_type}, {admission_type}, {semester}")

    # 1. Recognize different application type phrases
    app_aliases = {
        "first-year": "FY_APP_DEADLINES",
        "first year": "FY_APP_DEADLINES",
        "freshman": "FY_APP_DEADLINES",
        "undergraduate": "FY_APP_DEADLINES",
        "fy": "FY_APP_DEADLINES",
        "transfer": "TRANSFER_APP_DEADLINES",
        "transfer student": "TRANSFER_APP_DEADLINES"
    }

    # 2. Normalize application type
    app_input = application_type.strip().lower()
    app_key = app_aliases.get(app_input, app_input.upper())

    # 3. Normalize admission type
    admission_key = admission_type.strip().title()

    # 4. Normalize semester
    semester_key = semester.strip().lower()

    if not semester_key.endswith("semester"):
        semester_key += " semester"

    # 5. Validate application type
    if app_key not in APP_DEADLINES:
        return f"Application type '{application_type}' was not found."

    # 6. Validate admission type
    if admission_key not in APP_DEADLINES[app_key]:
        return f"Admission type '{admission_type}' was not found."

    # 7. Validate semester
    deadlines = APP_DEADLINES[app_key][admission_key]

    if semester_key not in deadlines:
        return f"No deadline found for {semester}."

    # 8. Return deadline
    deadline = deadlines[semester_key]

    return (
        f"The {admission_key} deadline for the "
        f"{application_type} {semester} is {deadline}."
    )
#Base on user's interest, recommend top 3 related programs

#Record the user's interest in a Lead.txt file (ask for name, email, and area of interests  or programs.)

#Provide applicationd deadlines and requirements.

#If user ask about Graduate program, explain that it would be best for them to contact the specific program because
#every graduate program is a little different. Handoff to UC directory agent.

In [ ]:
uc_admissions_instructions = f"""{RECOMMENDED_PROMPT_PREFIX}
You are a helpful admissions assistant that can help answer questions about applications,
programs, campus visits, and connect students and parents to the admissions counselors.

   
"""
uc_admissions_agent = Agent(
    name="UC Admissions Agent",
    instructions=uc_admissions_instructions,
    model=azure_model,
    tools=[uc_directory_search],
)

# Orchestrator

In [ ]:
directory_tool = uc_directory_agent.as_tool(
    tool_name="consult_uc_directory",
    tool_description="Look up UC department phone numbers, emails, and websites.",
)

orchestrator_instructions = f"""{RECOMMENDED_PROMPT_PREFIX}
You are UC New Bearcat Helper, a helpful, caring, and professional counselor that helps
answer questions and inquiries from new students and parents. Your main goal is to
provide information about the University of Cincinnati and guide them to the right resources.

You will help students and parents understand more about UC (tour information, admission
requirements, tuition, how to apply, etc.). Encourage them to check program websites.

When the user asks for a department contact, phone number, email, or website, call
consult_uc_directory and use only the details it returns. Do not invent contact details.
If a single message needs a directory lookup plus other guidance, consult the directory
tool and combine everything into one clear reply.
If the directory tool does not find a match, direct the student to the Automated Directory
Attendant at (513) 556-6000.
"""

orchestrator = Agent(
    name="UC New Bearcat Helper",
    instructions=orchestrator_instructions,
    model=azure_model,
    tools=[directory_tool],
)

print("Agent built:", orchestrator.name)
print("Tools available:", [t.name for t in orchestrator.tools])

3. Gradio Chat

In [ ]:
chat_session = SQLiteSession("gradio_student", "chat_history.db")

async def chat(message, history):
    result = await Runner.run(orchestrator, message, session=chat_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)